[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C39_Distributed_Training_Course/02_data_parallel_fsdp/02_data_parallel_fsdp.ipynb)

# 02 · 数据并行与 FSDP（用 numpy 在单进程模拟多 rank）

本环境**没有多 GPU、没有 `torch.distributed`**。我们用一条核心技巧把分布式训练搬到单进程 numpy 里：

> **把「分布在 W 个 rank 上的张量」表示成一个长度为 W 的 Python 列表**（`shards[r]` = rank r 那一份）。
> 于是 **all-reduce(sum) 就是把列表里的数组相加**，**reduce-scatter / all-gather** 就是切分与拼接列表。

路线：DDP 梯度平均 ≡ 单卡大 batch → ZeRO 各阶段每 rank 显存账 → reduce-scatter+all-gather ≡ all-reduce → FSDP 前向(all-gather 参数) ≡ 单卡前向 → 优化器分片更新 → ✏️ 练习 → 📖 答案 → 🧪 真实模型(Llama-2-7B)显存胶囊 → 🔧 PyTorch DDP/FSDP 真实代码对照。

**纪律**：每个「分布式实现」都和一个**单卡参考**对拍，标准 `np.allclose(..., atol=1e-9)`。结构对 → 数值一致 → 逻辑可迁移到真实 FSDP。

## 0 · 热身：把集合通信当成对列表的操作

复用模块 01 的视角，但这里我们让每个 rank 持有的是**模型梯度/参数**。先把三个原语写成最朴素的列表操作，后面反复用。

In [ ]:
import numpy as np
rng = np.random.default_rng(39)

def all_reduce_sum(shards):
    '''输入: 每个 rank 一个数组的列表; 输出: 每个 rank 都拿到的「逐元素求和」结果。'''
    total = np.sum(shards, axis=0)        # 模拟跨 rank 求和
    return [total.copy() for _ in shards] # all-reduce 后人人一份相同结果

def reduce_scatter_sum(shards):
    '''先逐元素求和，再把结果按 rank 数均分，每个 rank 只拿自己那一片。'''
    W = len(shards)
    total = np.sum(shards, axis=0)
    pieces = np.array_split(total, W)     # 切成 W 片
    return [pieces[r].copy() for r in range(W)]

def all_gather(local_pieces):
    '''每个 rank 贡献自己那片, 拼成完整数组, 人人一份。'''
    full = np.concatenate(local_pieces)
    return [full.copy() for _ in local_pieces]

# 自检：all-reduce == 各 rank 数组之和
W = 4
g = [rng.standard_normal(8) for _ in range(W)]
ar = all_reduce_sum(g)
assert all(np.allclose(x, sum(g)) for x in ar)
print('✅ all_reduce_sum 正确；W =', W, '，每 rank 都拿到相同的求和结果')

## 1 · DDP 核心等式：平均梯度 ≡ 单卡大 batch 的梯度

用一个最小的线性回归：`ŷ = X @ w`，损失 `L = (1/N)·||Xw − y||²`（对样本求平均）。
它的梯度有闭式：`g = (2/N)·Xᵀ(Xw − y)`。

我们把 N 个样本切给 W 个 rank，各算**本地平均梯度**，再 all-reduce 求平均，验证它**逐位等于**单卡在全部 N 个样本上的梯度。这就是 DDP 正确性的全部基石。

In [ ]:
D = 5                 # 特征维
W = 4                 # rank 数
b = 3                 # 每 rank 本地 batch
N = W * b             # 全局 batch
w = rng.standard_normal(D)
X = rng.standard_normal((N, D))
y = rng.standard_normal(N)

def grad_mse(Xb, yb, w):
    '''逐样本平均 MSE 的梯度: (2/n) Xᵀ(Xw − y)。'''
    n = Xb.shape[0]
    resid = Xb @ w - yb
    return (2.0 / n) * Xb.T @ resid

# 单卡参考：在全部 N 个样本上算
g_single = grad_mse(X, y, w)

# DDP：切成 W 片，各算本地平均梯度
Xs = np.array_split(X, W); ys = np.array_split(y, W)
local_grads = [grad_mse(Xs[r], ys[r], w) for r in range(W)]
# all-reduce 求和后除以 W = 求平均
g_ddp = all_reduce_sum(local_grads)[0] / W

print('单卡梯度 g_single[:3] =', np.round(g_single[:3], 5))
print('DDP 平均 g_ddp[:3]    =', np.round(g_ddp[:3], 5))
assert np.allclose(g_ddp, g_single, atol=1e-12), 'DDP 平均梯度必须等于单卡大 batch 梯度'
print('✅ DDP 等式成立：每 rank 本地平均梯度再求平均 == 单卡 N 样本梯度')
print('   (前提：每 rank 本地 batch 相等；不等时需按样本数加权，见练习)')

## 2 · ZeRO 各阶段每 rank 显存账

混合精度 + Adam 下，常驻显存（不含激活）= **16P 字节**：fp16 参数 2P + fp16 梯度 2P + fp32 主参数 4P + Adam m 4P + Adam v 4P。

把每个 ZeRO 阶段「哪些项被切成 1/W」编码成一个函数，验证 **0 > 1 > 2 > 3 单调递减**。

In [ ]:
def per_rank_bytes(P, W, stage):
    '''返回单 rank 常驻显存(字节, 不含激活)。
       项: fp16参数2P, fp16梯度2P, fp32主参4P, Adam m 4P, Adam v 4P (合计16P)。'''
    params  = 2 * P                      # fp16 参数
    grads   = 2 * P                      # fp16 梯度
    optim   = 4 * P + 4 * P + 4 * P      # fp32 主参 + m + v = 12P
    if stage == 0:                       # DDP: 全量
        pass
    elif stage == 1:                     # 分片优化器状态
        optim /= W
    elif stage == 2:                     # 再分片梯度
        optim /= W; grads /= W
    elif stage == 3:                     # 再分片参数 (FSDP)
        optim /= W; grads /= W; params /= W
    else:
        raise ValueError(stage)
    return params + grads + optim

P = 7e9      # 7B 参数
W = 8
GB = 1024**3
print(f'{"阶段":<10}{"每 rank 显存(GB)":>18}')
vals = []
for s in range(4):
    gb = per_rank_bytes(P, W, s) / GB
    vals.append(gb)
    print(f'ZeRO-{s:<6}{gb:>18.1f}')
assert vals[0] > vals[1] > vals[2] > vals[3], '显存必须随 stage 单调下降'
assert abs(vals[0] - 16*P/GB) < 1e-6, 'ZeRO-0 应为 16P'
assert abs(vals[3] - 16*P/W/GB) < 1e-6, 'ZeRO-3 应为 16P/W'
print('✅ 16P → 16P/W：ZeRO 三阶段逐级把冗余切掉')

## 3 · 关键恒等式：reduce-scatter + all-gather ≡ all-reduce

DDP 的一次 all-reduce，在 ZeRO 里被拆成**反向后 reduce-scatter 梯度** + **更新后 all-gather 参数**两步。
高效的 ring all-reduce 内部本就是这两步（模块 01）。这里验证：先 reduce-scatter 再 all-gather，结果**等于**直接 all-reduce。

In [ ]:
W = 4
shards = [rng.standard_normal(12) for _ in range(W)]   # 每 rank 一份完整梯度

# 路线 A：直接 all-reduce（求和）
ref = all_reduce_sum(shards)[0]

# 路线 B：reduce-scatter（每 rank 只留 1/W 求和分片）→ all-gather（拼回完整）
pieces = reduce_scatter_sum(shards)                     # 每 rank: 自己那 1/W 片的和
gathered = all_gather(pieces)[0]                        # 拼回完整

print('all-reduce 结果[:4]            =', np.round(ref[:4], 4))
print('reduce-scatter+all-gather[:4] =', np.round(gathered[:4], 4))
assert np.allclose(ref, gathered, atol=1e-12)
# 通信量直觉：两步各搬约 (W-1)/W · |g|，合计 ≈ 2|g|，与 all-reduce 相同
print('✅ 恒等式成立：这就是 ZeRO-1 把 DDP 的 all-reduce 拆成两步而通信量不变的原因')

## 4 · FSDP 前向：all-gather 分片参数 ≡ 单卡完整前向

FSDP 平时每 rank 只存 1/W 的参数；前向到某层时临时 all-gather 凑齐、算完丢弃。
验证：把一个权重矩阵按行分片到 W 个 rank，前向时 all-gather 还原，得到的输出**等于**用完整权重的单卡前向。

In [ ]:
d_in, d_out = 6, 8
W = 4
Wt = rng.standard_normal((d_out, d_in))    # 完整权重 (PyTorch Linear: y = x @ Wᵀ)
x  = rng.standard_normal((3, d_in))        # batch=3

# 单卡参考前向
y_ref = x @ Wt.T

# FSDP：按「输出维(行)」把 Wt 分片到各 rank，每 rank 只存 d_out/W 行
shards = np.array_split(Wt, W, axis=0)      # 每片形状 (d_out/W, d_in)
assert sum(s.shape[0] for s in shards) == d_out

def fsdp_layer_forward(x, weight_shards):
    # all-gather：把各 rank 的行分片拼回完整权重（模拟临时凑齐）
    Wt_full = np.concatenate(weight_shards, axis=0)
    y = x @ Wt_full.T
    del Wt_full                            # 用完即弃（模拟 free）
    return y

y_fsdp = fsdp_layer_forward(x, shards)
print('单卡输出 shape', y_ref.shape, ' FSDP 输出 shape', y_fsdp.shape)
assert np.allclose(y_fsdp, y_ref, atol=1e-12)
print('✅ FSDP 前向 == 单卡前向：分片只改变「参数存哪」，不改变「算什么」')

## 5 · 优化器分片更新：各 rank 更新自己那片 ≡ 单卡整体更新

ZeRO 里每个 rank 只持有 1/W 的优化器状态，**只更新自己负责的那 1/W 参数**，最后 all-gather 拼回完整参数。
用一步 SGD（`w ← w − lr·g`）验证：分片更新后 all-gather 的参数，等于单卡对整组参数做同一步更新。

In [ ]:
P_dim = 12          # 参数个数(玩具)
W = 4
lr = 0.1
w_full = rng.standard_normal(P_dim)
g_full = rng.standard_normal(P_dim)      # 假设已 all-reduce 好的完整平均梯度

# 单卡参考：整体更新
w_ref = w_full - lr * g_full

# ZeRO：每 rank 只更新自己负责的 1/W 片
w_shards = np.array_split(w_full, W)
g_shards = np.array_split(g_full, W)      # reduce-scatter 后每 rank 的梯度片
updated_pieces = [w_shards[r] - lr * g_shards[r] for r in range(W)]
# all-gather 拼回完整参数供下一步前向
w_zero = np.concatenate(updated_pieces)

assert np.allclose(w_zero, w_ref, atol=1e-12)
print('✅ 分片更新 == 整体更新：每 rank 只动 1/W 参数，拼回后逐位一致')
print('   每 rank 只需存 1/W 的优化器状态 ->', P_dim, '→', len(updated_pieces[0]), '/rank')

---
## ✏️ 练习 1：带权重的 all-reduce 梯度平均

现实中各 rank 的本地 batch 不一定相等（最后一个 batch、变长数据）。此时**简单平均梯度是错的**，必须按样本数加权。

实现 `weighted_grad_average(local_grads, counts)`：`local_grads[r]` 是 rank r 的**本地平均**梯度，`counts[r]` 是其样本数。
返回全局平均梯度（应等于把所有样本拼起来的单卡梯度）。提示：先把本地平均还原成「梯度和」再除以总样本数。

In [ ]:
def weighted_grad_average(local_grads, counts):
    # TODO: rank r 的「梯度和」= local_grads[r] * counts[r]
    #       全局平均 = Σ(梯度和) / Σ(counts)
    #       用 all_reduce_sum 模拟跨 rank 求和
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
D = 5
counts = [3, 5, 2, 4]            # 各 rank 样本数不等
Ntot = sum(counts)
w = rng.standard_normal(D)
X = rng.standard_normal((Ntot, D)); y = rng.standard_normal(Ntot)
g_single = grad_mse(X, y, w)    # 单卡参考
# 按 counts 切分
idx = np.cumsum(counts)[:-1]
Xs = np.split(X, idx); ys = np.split(y, idx)
local = [grad_mse(Xs[r], ys[r], w) for r in range(len(counts))]
g_w = weighted_grad_average(local, counts)
assert np.allclose(g_w, g_single, atol=1e-12), '加权平均应等于单卡全样本梯度'
# 反证：简单平均在 batch 不等时是错的
g_naive = np.mean(local, axis=0)
assert not np.allclose(g_naive, g_single, atol=1e-6), 'batch 不等时简单平均应当偏离'
print('✅ 练习 1 通过：batch 不等时必须按样本数加权，简单平均会错')

## ✏️ 练习 2：ZeRO 显存账与「装得下吗」

给定模型参数量 `P`、单卡显存 `mem_gb`、激活占用 `act_gb`，实现 `min_world_size(P, mem_gb, act_gb, stage)`：
返回该 ZeRO 阶段下**至少需要多少块卡**才能装下（每卡常驻 + 激活 ≤ mem_gb）。

提示：复用 `per_rank_bytes`，对 W = 1,2,3,... 递增试，找到第一个满足的 W。stage 0/1 的参数项不随 W 缩小，可能永远装不下 → 返回 `None`。

In [ ]:
def min_world_size(P, mem_gb, act_gb, stage, max_W=1024):
    # TODO: 对 W in 1..max_W: 若 per_rank_bytes(P,W,stage)/GB + act_gb <= mem_gb 返回 W
    #       都不满足返回 None
    GB = 1024**3
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
P = 7e9; mem = 80.0; act = 10.0      # 7B 模型, 80GB 卡, 10GB 激活
# ZeRO-0/1 参数+梯度全量 = 4P ≈ 26GB，加激活，单卡其实能放下 stage0? 16P=104GB>80 → 需要多卡
w0 = min_world_size(P, mem, act, stage=0)
w3 = min_world_size(P, mem, act, stage=3)
assert w3 is not None and w3 >= 1
assert w0 is None or w0 >= w3, 'stage0 要么装不下，要么需要不少于 stage3 的卡数'
# stage3 显存随 W 线性下降，一定能找到可行 W
assert per_rank_bytes(P, w3, 3)/(1024**3) + act <= mem + 1e-6
print(f'ZeRO-0 需要 {w0} 卡, ZeRO-3 需要 {w3} 卡（7B/80GB/10GB激活）')
print('✅ 练习 2 通过：ZeRO-3 显存随卡数下降，总能凑够；低 stage 可能怎么加卡都装不下')

## ✏️ 练习 3：FSDP shard / all-gather 往返

实现一对函数模拟 FSDP 对一个参数张量的**扁平化分片**（真实 FSDP 把参数 flatten 后按 1/W 切，可能需要 padding）：
- `fsdp_shard(param, W)`：把 1D 参数 pad 到 W 的整数倍后切成 W 片，返回 `(shards, orig_len)`；
- `fsdp_all_gather(shards, orig_len)`：拼回并去掉 padding，还原原参数。

验证 `all_gather(shard(p)) == p`，且每片长度相等（真实 all-gather 要求等长）。

In [ ]:
def fsdp_shard(param, W):
    # TODO: pad_len = (-len(param)) % W; 补零到整除; np.array_split 成 W 等长片
    #       返回 (shards, 原始长度)
    raise NotImplementedError

def fsdp_all_gather(shards, orig_len):
    # TODO: concatenate 后切片到 orig_len
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
for L in [12, 13, 17, 100]:        # 含不能整除 W 的情况
    p_ = rng.standard_normal(L)
    W = 4
    shards, orig = fsdp_shard(p_, W)
    assert len(shards) == W
    assert len(set(len(s) for s in shards)) == 1, '所有分片必须等长(all-gather 要求)'
    back = fsdp_all_gather(shards, orig)
    assert np.allclose(back, p_, atol=1e-12), f'L={L} 往返应还原'
print('✅ 练习 3 通过：flatten+pad 分片，all-gather 往返无损，且分片等长')

## ✏️ 练习 4：通信量——ZeRO-3 比 DDP 多 1.5×

按「每步每 rank 等效收发字节数」估算通信量并对比：
- DDP：反向后一次 all-reduce ≈ `2·(W−1)/W · P_bytes`（ring all-reduce：reduce-scatter + all-gather 各 `(W−1)/W·P`）；
- ZeRO-3：前向 all-gather 参数 + 反向 all-gather 参数 + reduce-scatter 梯度 ≈ `3·(W−1)/W · P_bytes`。

实现 `comm_volume(P_bytes, W, scheme)`，验证 `ZeRO-3 / DDP → 1.5`。

In [ ]:
def comm_volume(P_bytes, W, scheme):
    # TODO: factor = (W-1)/W
    #       'ddp'   -> 2 * factor * P_bytes   (all-reduce)
    #       'zero3' -> 3 * factor * P_bytes   (2 次 all-gather + 1 次 reduce-scatter)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
Pb = 7e9 * 2          # 7B 参数, fp16 -> 2 字节/参数
for W in [4, 8, 64, 512]:
    ddp = comm_volume(Pb, W, 'ddp')
    z3  = comm_volume(Pb, W, 'zero3')
    ratio = z3 / ddp
    assert abs(ratio - 1.5) < 1e-9, f'ZeRO-3/DDP 应恒为 1.5, 得到 {ratio}'
print('✅ 练习 4 通过：ZeRO-3 通信量恒为 DDP 的 1.5×（前向多一次 all-gather 参数）')
print('   这就是「显存省 → 通信多」权衡的定量形式')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def weighted_grad_average(local_grads, counts):
    sums = [local_grads[r] * counts[r] for r in range(len(counts))]
    total = all_reduce_sum(sums)[0]      # Σ 梯度和
    return total / sum(counts)           # 除以总样本数

In [ ]:
# 练习 2 参考答案
def min_world_size(P, mem_gb, act_gb, stage, max_W=1024):
    GB = 1024**3
    for W in range(1, max_W + 1):
        if per_rank_bytes(P, W, stage) / GB + act_gb <= mem_gb:
            return W
    return None

In [ ]:
# 练习 3 参考答案
def fsdp_shard(param, W):
    pad = (-len(param)) % W
    padded = np.concatenate([param, np.zeros(pad, dtype=param.dtype)])
    shards = list(np.array_split(padded, W))   # 等长
    return shards, len(param)

def fsdp_all_gather(shards, orig_len):
    return np.concatenate(shards)[:orig_len]

In [ ]:
# 练习 4 参考答案
def comm_volume(P_bytes, W, scheme):
    factor = (W - 1) / W
    if scheme == 'ddp':
        return 2 * factor * P_bytes
    elif scheme == 'zero3':
        return 3 * factor * P_bytes
    raise ValueError(scheme)

---
## 🧪 真实数据胶囊：Llama-2-7B 在不同 ZeRO 阶段要几块卡

用**真实模型配置**算一笔工程账：联网读取 Llama-2 的真实 `config.json`(用非 gated 的 `NousResearch/Llama-2-7b-hf`，与 Meta 官方 7B 同结构)，从 `hidden_size`/`num_hidden_layers`/`num_attention_heads`/`vocab_size` **精确算出参数量**(约 6.7e9)。**联网失败则回退**到这个真实公开数值，然后算各 ZeRO 阶段在 80GB(A100) / 40GB 卡上的可行卡数。

In [ ]:
# 真实模型配置（带联网回退；本环境通常离线 -> 用内置真实数值）
def llama_params_from_config(cfg):
    '''由 config 字段精确估算 Llama 参数量(标准 LLaMA 结构)。'''
    h   = cfg['hidden_size']                                   # 4096
    L   = cfg['num_hidden_layers']                             # 32
    V   = cfg['vocab_size']                                    # 32000
    ffn = cfg.get('intermediate_size', 4 * h)                 # 11008 (SwiGLU)
    kvh = cfg.get('num_key_value_heads', cfg['num_attention_heads'])
    head = h // cfg['num_attention_heads']
    embed = V * h                                              # token 嵌入(与输出层不共享)
    attn  = h * h + 2 * (kvh * head) * h + h * h               # q + k,v(可 GQA) + o
    mlp   = 3 * h * ffn                                        # gate + up + down (SwiGLU)
    per_layer = attn + mlp                                     # norm 量级可忽略
    return embed * (1 if cfg.get('tie_word_embeddings') else 2) + L * per_layer

def get_llama2_7b_params():
    try:
        import urllib.request, json
        # 非 gated 等价配置(与 meta-llama/Llama-2-7b-hf 同结构)
        url = 'https://huggingface.co/NousResearch/Llama-2-7b-hf/resolve/main/config.json'
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=8) as f:
            cfg = json.load(f)
        P = float(llama_params_from_config(cfg))               # 从真实 config 算参数量
        print(f'  读到真实 config: hidden={cfg["hidden_size"]} layers={cfg["num_hidden_layers"]} '
              f'heads={cfg["num_attention_heads"]} vocab={cfg["vocab_size"]}')
        return P, 'NousResearch/Llama-2-7b-hf config.json (online, 非 gated)'
    except Exception as e:
        print('  (联网失败，回退真实公开值:', type(e).__name__, ')')
        return 6.7e9, 'offline-fallback(真实公开值)'

P, src = get_llama2_7b_params()
print(f'Llama-2-7B 参数量 ≈ {P:.3e}  (来源: {src})')
assert 6.0e9 < P < 7.5e9, '7B 量级(从 config 算出的参数量应落在 6.0e9~7.5e9)'
print(f'混合精度+Adam 总显存 16P = {16*P/(1024**3):.0f} GB  -> 单张 80GB 卡装不下！')

**🧪 胶囊练习**：实现 `feasible_table(P, cards)`：对一组卡显存配置和 ZeRO 阶段，打印每种组合至少需要的卡数（激活按 8GB 估）。
复用 `min_world_size`。下面骨架与自测留给你补全。

In [ ]:
def feasible_table(P, mem_list=(40, 80), act_gb=8.0):
    # TODO: 对每个 mem in mem_list、每个 stage in [0,1,2,3]，
    #       用 min_world_size 求最少卡数并打印成表；装不下打印 '装不下'
    raise NotImplementedError

In [ ]:
# 🧪 胶囊自测
feasible_table(P)
# 校验：80GB 上 ZeRO-3 一定可行，且不多于 40GB 上 ZeRO-3 的卡数
w80 = min_world_size(P, 80, 8.0, 3)
w40 = min_world_size(P, 40, 8.0, 3)
assert w80 is not None and w40 is not None and w80 <= w40
print(f'\n✅ 胶囊通过：80GB 卡 ZeRO-3 需 {w80} 卡，40GB 需 {w40} 卡（显存越小越费卡）')

In [ ]:
# 📖 胶囊参考答案
def feasible_table(P, mem_list=(40, 80), act_gb=8.0):
    print(f'{"卡显存":>8}{"ZeRO-0":>10}{"ZeRO-1":>10}{"ZeRO-2":>10}{"ZeRO-3":>10}')
    for mem in mem_list:
        row = [f'{mem}GB'.rjust(8)]
        for s in range(4):
            w = min_world_size(P, mem, act_gb, s)
            row.append(('装不下' if w is None else f'{w}卡').rjust(10))
        print(''.join(row))

---
## 🔧 旁注：真实 PyTorch 里的 DDP 与 FSDP（对照，不在本环境跑）

我们 numpy 模拟的「梯度 all-reduce」「参数分片 all-gather」，在 PyTorch 里就是下面几行（伪代码）：

```python
import torch, torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

dist.init_process_group('nccl')            # 建立 world（rank/world_size）
model = MyTransformer().cuda()

# —— 数据并行：每卡一份完整模型，反向自动 all-reduce 平均梯度（= 我们 worked 1）
model = DDP(model, device_ids=[local_rank], gradient_as_bucket_view=True)

# —— FSDP / ZeRO-3：参数/梯度/优化器状态全分片，逐 unit all-gather（= 我们 worked 4/5）
model = FSDP(model, sharding_strategy=ShardingStrategy.FULL_SHARD,   # ZeRO-3
             auto_wrap_policy=transformer_auto_wrap_policy)           # 按 block 包装成 FSDP unit
# ShardingStrategy: SHARD_GRAD_OP≈ZeRO-2, HYBRID_SHARD=HSDP（节点内分片+节点间复制）
```

对应关系：`DDP` 的反向钩子 ↔ 我们的 `all_reduce_sum`/W；`FSDP` 的 `_all_gather` ↔ 我们的 `fsdp_all_gather`；`reduce_scatter` ↔ 我们的 `reduce_scatter_sum`。你在 numpy 里验证过的分片/聚合逻辑，与真实实现**结构一一对应**。

### 小结
- **数据并行 = 复制模型 + 切分数据 + 每步 all-reduce 平均梯度**；其正确性是一条等式：平均梯度 ≡ 单卡大 batch 梯度。
- 混合精度+Adam 常驻显存 **16P**；DDP 让每卡冗余存 12P 优化器状态——这是浪费的根源。
- **ZeRO-1/2/3** 逐级分片优化器状态→梯度→参数，每 rank 显存 16P → **16P/W**；代价是通信增多（ZeRO-3 ≈ 1.5× DDP）。
- **reduce-scatter + all-gather ≡ all-reduce** 是把 DDP 拆成 ZeRO 的代数基石。
- 选型次序：**先用显存筛掉装不下的，再在能装下的里挑通信最省的**；HSDP 利用节点内/外带宽差。

下一站：**模块 03 · 张量与流水并行** —— 当单层本身就大到一张卡放不下，数据并行/分片也无能为力，必须把单个算子切开。